# WAXAL ASR — full pipeline (Kaggle GPU)

MMS-1b + Whisper-turbo + KenLM + LID ensemble for Luganda / Lingala / Shona.
Set **Accelerator = GPU (P100 or T4 x2)** and **Internet = ON**.
Attach the Zindi challenge files as a dataset at `/kaggle/input/waxal-challenge`.
See `STRATEGY.md` for the reasoning and `README.md` for details.

## 0. Get the code + install deps

The repo `riaz22/waxal-asr` is **public**, so this cell just clones it (make sure
Internet is ON in Settings). If it's already cloned it does a `git pull` to grab
your latest push. Fallback: attach the repo as a Kaggle dataset under
`/kaggle/input` and the cell will find it there.

In [ ]:
import os, sys, subprocess, glob, shutil
REPO = "/kaggle/working/waxal-asr"
GH = "riaz22/waxal-asr"

def find_code_root(base):
    hits = glob.glob(os.path.join(base, "**", "src", "config.py"), recursive=True)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None

root = find_code_root(REPO)
if root:                                           # already here -> update it
    subprocess.run(["git", "-C", root, "pull", "--ff-only"], check=False)
else:                                              # clone the public repo
    try:
        subprocess.run(["git", "clone",
                        f"https://github.com/{GH}.git", REPO], check=True)
    except Exception:                              # fallback: attached dataset
        ds_root = find_code_root("/kaggle/input")
        if ds_root:
            shutil.copytree(ds_root, REPO, dirs_exist_ok=True)
    root = find_code_root(REPO)

if root is None:
    raise SystemExit(
        "Repo not found. Turn Internet ON to clone github.com/" + GH +
        ", or attach it as a dataset under /kaggle/input, then re-run.")
sys.path.insert(0, root); os.chdir(root)
print("code root:", root); print(sorted(os.listdir(root)))

In [ ]:
# Kaggle already ships transformers (v5), datasets, accelerate, torch,
# soundfile, peft. Install ONLY the extras -- and DO NOT install librosa: it
# pulls numba which downgrades numpy and breaks the preinstalled pandas
# ("numpy.dtype size changed" ABI error). We read audio via datasets.Audio
# (soundfile backend), so librosa isn't needed.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "jiwer", "pyctcdecode"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "https://github.com/kpu/kenlm/archive/master.zip"])
print("extras installed")

## 1. Prepare data — RUN THIS FIRST (everything else needs it)

Downloads the 3 ASR configs from HF, cleans them, and writes the cleaned splits
+ vocab + LM corpus to `/kaggle/working/waxal-clean`. **This takes a while (tens
of minutes; downloads several GB)** — let it finish before running any later
cell. The challenge CSVs ship inside the repo, so nothing else needs attaching.

Optional (saves time next session): after this finishes, save the notebook's
`/kaggle/working` output as a Kaggle Dataset named **`waxal-clean`** and attach it
to future runs — the code auto-detects it and skips re-downloading.

In [ ]:
!python scripts/prepare_data.py

## 2. Build KenLM 5-gram per language

In [ ]:
%%capture
!apt-get -qq install -y build-essential cmake libboost-all-dev libeigen3-dev zlib1g-dev
!git clone -q https://github.com/kpu/kenlm.git /kaggle/working/kenlm
!cd /kaggle/working/kenlm && mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release >/dev/null && make -j4 >/dev/null

In [ ]:
!python scripts/build_lm.py --kenlm_bin /kaggle/working/kenlm/build/bin

## 3. Train MMS-1b adapters (one per language)

In [ ]:
!python scripts/train_mms.py --lang lin

In [ ]:
!python scripts/train_mms.py --lang sna

In [ ]:
!python scripts/train_mms.py --lang lug

## 4. Train joint Whisper-turbo (add --lora if you OOM)

In [ ]:
!python scripts/train_whisper.py

## 5. Language ID (validate on Phase 1, run on test)

In [ ]:
!python scripts/lid.py --split validation

In [ ]:
!python scripts/lid.py --split test

## 6. Validate ensemble locally BEFORE submitting

In [ ]:
!python scripts/infer.py --model mms     --split validation
!python scripts/infer.py --model whisper --split validation
!python scripts/ensemble.py --split validation

## 7. Predict test + build submission

In [ ]:
!python scripts/infer.py --model mms     --split test
!python scripts/infer.py --model whisper --split test
!python scripts/ensemble.py --split test --route artifacts/route.json

In [ ]:
import pandas as pd
sub = pd.read_csv("artifacts/submission.csv"); print(sub.shape); sub.head()